In [34]:
# =============================================================================
# GRÁFICO REGRESIÓN CON LAGS - Mercado + Chacutería + Competidores
# =============================================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from datetime import timedelta

# ====================== CARGA DE DATOS ======================
df_merc = pd.read_csv('datos_limpios/mercado.csv')
df_out  = pd.read_csv('datos_limpios/consolidado_sell_out.csv')

df_merc['mes'] = pd.to_datetime(df_merc['mes'])
df_out['fecha_factura'] = pd.to_datetime(df_out['fecha_factura'], errors='coerce')

# ====================== FILTROS (cambia aquí) ======================
canal_sel     = 'Grandes Superficies'   # ← Cambia o pon None
categoria_sel = 'CARNES FRIAS'          # ← Cambia o pon None

if canal_sel and categoria_sel:
    df_m = df_merc[(df_merc['canal'] == canal_sel) & 
                   (df_merc['categoria'] == categoria_sel)].copy()
else:
    df_m = df_merc.copy()

# Calcular volúmenes reales
df_m['vol_chacuteria'] = df_m['ventas_mercado_volumen'] * df_m['share_compania']
df_m['vol_comp1']      = df_m['ventas_mercado_volumen'] * df_m['share_competidor_1']
df_m['vol_comp2']      = df_m['ventas_mercado_volumen'] * df_m['share_competidor_2']

# Agrupación mensual
hist = df_m.groupby('mes').agg({
    'ventas_mercado_volumen': 'sum',
    'vol_chacuteria': 'sum',
    'vol_comp1': 'sum',
    'vol_comp2': 'sum',
    'ventas_mercado_valor': 'sum'
}).reset_index()

# ====================== REGRESIÓN CON LAGS ======================
data_reg = hist[['mes', 'ventas_mercado_volumen']].copy()
data_reg = data_reg.rename(columns={'mes': 'ds', 'ventas_mercado_volumen': 'y'})

# Crear lags
for lag in [1, 2, 3]:
    data_reg[f'lag_{lag}'] = data_reg['y'].shift(lag)
data_reg = data_reg.dropna().reset_index(drop=True)

X = data_reg[['lag_1', 'lag_2', 'lag_3']]
y = data_reg['y']

model = LinearRegression()
model.fit(X, y)

# Pronóstico 6 meses adelante
ult_lags = data_reg[['lag_1', 'lag_2', 'lag_3']].iloc[-1:].values.copy()
pron_market = []
fechas_fut = pd.date_range(start=hist['mes'].max() + timedelta(days=30), periods=6, freq='M')

for _ in range(6):
    pred = model.predict(ult_lags)[0]
    pron_market.append(max(0, pred))
    ult_lags = np.roll(ult_lags, 1)
    ult_lags[0][0] = pred

# Derivar pronósticos de Chacutería y competidores (usando share promedio histórico)
avg_share_ch = hist['vol_chacuteria'].mean() / hist['ventas_mercado_volumen'].mean()
avg_share_c1 = hist['vol_comp1'].mean()      / hist['ventas_mercado_volumen'].mean()
avg_share_c2 = hist['vol_comp2'].mean()      / hist['ventas_mercado_volumen'].mean()

pron_chac = np.array(pron_market) * avg_share_ch
pron_comp1 = np.array(pron_market) * avg_share_c1
pron_comp2 = np.array(pron_market) * avg_share_c2

# ====================== GRÁFICO INTERACTIVO (igual estilo que Prophet) ======================
fig = go.Figure()

# Mercado
fig.add_trace(go.Scatter(x=hist['mes'], y=hist['ventas_mercado_volumen'], 
                         name='Mercado Histórico', line=dict(color='#2ec4b6', width=3)))
fig.add_trace(go.Scatter(x=fechas_fut, y=pron_market, 
                         name='Mercado Pronóstico', line=dict(color='#ff6b35', dash='dash', width=4)))

# Chacutería
fig.add_trace(go.Scatter(x=hist['mes'], y=hist['vol_chacuteria'], 
                         name='Chacutería Histórico', line=dict(color='#57cc99')))
fig.add_trace(go.Scatter(x=fechas_fut, y=pron_chac, 
                         name='Chacutería Pronóstico', line=dict(color='#57cc99', dash='dash')))

# Competidor 1
fig.add_trace(go.Scatter(x=hist['mes'], y=hist['vol_comp1'], 
                         name='Competidor 1 Histórico', line=dict(color='#ffd166')))
fig.add_trace(go.Scatter(x=fechas_fut, y=pron_comp1, 
                         name='Competidor 1 Pronóstico', line=dict(color='#ffd166', dash='dash')))

# Competidor 2
fig.add_trace(go.Scatter(x=hist['mes'], y=hist['vol_comp2'], 
                         name='Competidor 2 Histórico', line=dict(color='#a9b8d5')))
fig.add_trace(go.Scatter(x=fechas_fut, y=pron_comp2, 
                         name='Competidor 2 Pronóstico', line=dict(color='#a9b8d5', dash='dash')))

# Hover con valor del mercado
fig.update_traces(
    customdata=hist['ventas_mercado_valor'].tolist() + [0]*6,
    hovertemplate="<b>%{fullData.name}</b><br>Mes: %{x}<br>Volumen: %{y:,.0f} unidades<br>Valor Mercado: $%{customdata:,.0f}<extra></extra>"
)

fig.update_layout(
    title=f"Pronóstico Regresión con Lags - {canal_sel} | {categoria_sel}",
    xaxis_title="Mes",
    yaxis_title="Unidades de Volumen",
    template="plotly_dark",
    height=580,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.01)
)

fig.show()

# ====================== RESULTADOS NUMÉRICOS ======================
print("\n=== PRONÓSTICO REGRESIÓN CON LAGS (Próximos 6 meses) ===")
for f, mkt, chac, c1, c2 in zip(fechas_fut.strftime('%Y-%m'), pron_market, pron_chac, pron_comp1, pron_comp2):
    print(f"{f} → Mercado: {mkt:,.0f} | Chacutería: {chac:,.0f} | Comp1: {c1:,.0f} | Comp2: {c2:,.0f}")


=== PRONÓSTICO REGRESIÓN CON LAGS (Próximos 6 meses) ===
2025-12 → Mercado: 3,527 | Chacutería: 405 | Comp1: 699 | Comp2: 563
2026-01 → Mercado: 3,413 | Chacutería: 392 | Comp1: 677 | Comp2: 545
2026-02 → Mercado: 3,475 | Chacutería: 399 | Comp1: 689 | Comp2: 554
2026-03 → Mercado: 3,378 | Chacutería: 388 | Comp1: 670 | Comp2: 539
2026-04 → Mercado: 3,435 | Chacutería: 395 | Comp1: 681 | Comp2: 548
2026-05 → Mercado: 3,377 | Chacutería: 388 | Comp1: 669 | Comp2: 539


In [37]:


# ====================== 4. DOS KPIs DE SHARE (Top 3 y Bottom 3) ======================
print("\n" + "="*60)
print("📈 KPIs DE SHARE - CHACUTERÍA FOODS")
print("="*60)

# Agrupamos por canal + categoría
share_group = df.groupby(['canal', 'categoria'])['share_compania'].mean().reset_index()
share_group = share_group.sort_values('share_compania', ascending=False)

# Top 3
top3 = share_group.head(3).copy()
top3['share_%'] = (top3['share_compania'] * 100).round(2)
print("\n🔥 TOP 3 CANALES/CATEGORÍAS CON MAYOR SHARE")
print(top3[['canal', 'categoria', 'share_%']].to_string(index=False))

# Bottom 3
bottom3 = share_group.tail(3).copy()
bottom3['share_%'] = (bottom3['share_compania'] * 100).round(2)
print("\n⚠️ BOTTOM 3 CANALES/CATEGORÍAS CON MENOR SHARE")
print(bottom3[['canal', 'categoria', 'share_%']].to_string(index=False))


📈 KPIs DE SHARE - CHACUTERÍA FOODS

🔥 TOP 3 CANALES/CATEGORÍAS CON MAYOR SHARE
              canal       categoria  share_%
Grandes Superficies QUESO PARMESANO    17.45
Grandes Superficies  QUESOS FRESCOS    16.31
      Hard Discount QUESOS ANALOGOS    15.80

⚠️ BOTTOM 3 CANALES/CATEGORÍAS CON MENOR SHARE
     canal        categoria  share_%
E-commerce     CARNES FRIAS     5.57
E-commerce CARNES MADURADAS     5.54
E-commerce  QUESOS ANALOGOS     5.45
